In [97]:
from google.colab import drive
import sys

In [98]:
drive.mount('/content/drive')
sys.path.append(
"/content/drive/MyDrive/SupportAI"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [99]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from datasets import Dataset
import json

In [100]:
path = '/content/drive/MyDrive/SupportAI'
df = pd.read_csv(f'{path}/data/raw/merged_support_dialogues_client_only.csv')
df.head()

,Unnamed: 0,dialogue_id,turn,text,role,topic,intent,sentiment,urgency
0,0,1,1,Не могли бы вы подсказать: у меня не проходит ...,client,payment,payment_issue,negative,3
1,2,1,3,"Прошу прощения за беспокойство, карта та же, ч...",client,payment,payment_issue,negative,3
2,4,2,1,"Привет, деточка, не получается привязать новю ...",client,payment,complaint_issue,negative,4
3,6,2,3,"Ой, простите за беспокойство, ошибка появляетс...",client,payment,payment_issue,negative,4
4,8,2,5,"Вот веьд незадача... спасибо большое, буду жда...",client,payment,gratitude,positive,1


In [101]:
targets = ['topic', 'intent', 'sentiment']
for target in targets:
    if target not in df.columns:
        raise ValueError(f"Признак '{target}' не найден в DataFrame.")

#### Кодируем метки признаков

In [102]:
def encode_labels(df, column):
    classes = sorted(df[column].unique())

    label2id = {str(label): int(idx) for idx, label in enumerate(classes)}
    id2label = {int(idx): str(label) for idx, label in enumerate(classes)}

    df[f"{column}_label"] = df[column].map(label2id)

    return df, label2id, id2label

In [103]:
df, topic_label2id, topic_id2label = encode_labels(df, 'topic')
df, intent_label2id, intent_id2label = encode_labels(df, 'intent')
df, sentiment_label2id, sentiment_id2label = encode_labels(df, 'sentiment')

In [104]:
label_mappings = {
    "topic": {
        "label2id": (topic_label2id),
        "id2label": (topic_id2label),
    },
    "intent": {
        "label2id": (intent_label2id),
        "id2label": (intent_id2label),
    },
    "sentiment": {
        "label2id": (sentiment_label2id),
        "id2label": (sentiment_id2label),
    },
}

In [105]:
with open(f'{path}/label_mappings.json', 'w', encoding='utf-8') as f:
    json.dump(label_mappings, f, ensure_ascii=False, indent=4)

#### Смотрим распределение класса topic до разделения выборок

In [106]:
df.value_counts(subset=['topic'], normalize=True).mul(100).round(2).astype(str) + '%'

,proportion
topic,
return,14.56%
payment,14.5%
complaint,14.39%
subscription,14.34%
delivery,14.22%
account_access,14.2%
technical_issue,13.78%


#### Разделяем данные на обучающую, валидационную и тестовую выборки

In [107]:
dialogues = (
    df.groupby("dialogue_id", as_index=False)
      .first()[["dialogue_id", "topic"]]
)

In [108]:
train_dialogues, temp_dialogues = train_test_split(
    dialogues,
    test_size=0.2,
    random_state=42,
    stratify=dialogues["topic"],
    shuffle=True
)

val_dialogues, test_dialogues = train_test_split(
    temp_dialogues,
    test_size=0.5,
    random_state=42,
    stratify=temp_dialogues["topic"],
    shuffle=True
)

In [109]:
train_df = df[df["dialogue_id"].isin(train_dialogues["dialogue_id"])]
val_df = df[df["dialogue_id"].isin(val_dialogues["dialogue_id"])]
test_df = df[df["dialogue_id"].isin(test_dialogues["dialogue_id"])]

#### Проверяем размер и распределение topic в выборках

In [110]:
print(f"Размер обучающей выборки: {len(train_df)}")
print(f"Размер валидационной выборки: {len(val_df)}")
print(f"Размер тестовой выборки: {len(test_df)}")

Размер обучающей выборки: 5077
Размер валидационной выборки: 637
Размер тестовой выборки: 630


In [111]:
print(f"Распределение тем в обучающей выборке:\n{train_df['topic'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%'}")
print(f"Распределение тем в валидационной выборке:\n{val_df['topic'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%'}")
print(f"Распределение тем в тестовой выборке:\n{test_df['topic'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%'}")

Распределение тем в обучающей выборке:
topic
payment             14.6%
return             14.54%
complaint          14.42%
subscription       14.34%
account_access     14.18%
delivery           14.16%
technical_issue    13.77%
Name: proportion, dtype: object
Распределение тем в валидационной выборке:
topic
return             14.91%
payment             14.6%
subscription       14.29%
delivery           14.29%
complaint          13.97%
technical_issue    13.97%
account_access     13.97%
Name: proportion, dtype: object
Распределение тем в тестовой выборке:
topic
complaint           14.6%
delivery            14.6%
account_access      14.6%
subscription       14.44%
return             14.44%
technical_issue    13.65%
payment            13.65%
Name: proportion, dtype: object


#### Загружаем токенизатор, который использовался при предобучении модели RuBERT, с помощью него сегментируем наши обращения

In [112]:
model_name = "DeepPavlov/rubert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

##### Максимальный размер токена 31, поэтому хватит длины 64

In [113]:
max_length = 64
def tokenize(batch):
    return tokenizer(batch['text'],
                     padding='max_length',
                     truncation=True,
                     max_length=max_length)

#### Переводим выборки в нужный формат

In [114]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

#### Токенизируем обращения

In [115]:
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/5077 [00:00<?, ? examples/s]

Map:   0%|          | 0/637 [00:00<?, ? examples/s]

Map:   0%|          | 0/630 [00:00<?, ? examples/s]

##### Пример токенизации

In [116]:
sample = train_dataset[0]

print(sample)

{'Unnamed: 0': 0, 'dialogue_id': 1, 'turn': 1, 'text': 'Не могли бы вы подсказать: у меня не проходит оплата картой, уже пробовал несколько раз, номер заказа 256837.', 'role': 'client', 'topic': 'payment', 'intent': 'payment_issue', 'sentiment': 'negative', 'urgency': 3, 'topic_label': 3, 'intent_label': 6, 'sentiment_label': 0, 'urgency_label': None, '__index_level_0__': 0, 'input_ids': [101, 9257, 12795, 1655, 1761, 51280, 2110, 156, 875, 14198, 1699, 12455, 75133, 76303, 128, 4745, 74177, 5325, 2226, 128, 15024, 49621, 21987, 51136, 151, 132, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0

#### Переводим выборки в тензорный вид

In [117]:
columns = [
    "input_ids",
    "attention_mask",
    "topic_label",
    "intent_label",
    "sentiment_label",
    "urgency"
]

train_dataset.set_format(
    type="torch",
    columns=columns
)

val_dataset.set_format(
    type="torch",
    columns=columns
)

test_dataset.set_format(
    type="torch",
    columns=columns
)

#### Проверяем, что все успешно выполнилось

In [118]:
sample = train_dataset[0]

print(sample["input_ids"].shape)
print(sample["attention_mask"].shape)
print(sample["topic_label"])

torch.Size([64])
torch.Size([64])
tensor(3)


#### Сохраняем разделенные данные на диск

In [119]:
train_dataset.save_to_disk(f"{path}/data/processed/train")
val_dataset.save_to_disk(f"{path}/data/processed/val")
test_dataset.save_to_disk(f"{path}/data/processed/test")

Saving the dataset (0/1 shards):   0%|          | 0/5077 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/637 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/630 [00:00<?, ? examples/s]